### Topic modeling

In [25]:
!pip install bertopic
!pip install sentence-transformers

In [26]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import re

In [27]:
df_pelis = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/peliculas_limpio.csv")

In [28]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/usuarios.csv")

## Preprocesado

In [29]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

In [30]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + ". "
    # + df_pelis["director"].fillna('').apply(limpiar_texto) + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)

## Entrenamiento

In [31]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [32]:
embeddings = model.encode(df_pelis["texto"].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

In [45]:
# Instanciación con configuración mínima
topic_model = BERTopic(
    embedding_model=model,
    calculate_probabilities=True,
    verbose=True
    )

# Entrenamiento (pasando embeddings pre-computados)
topics, probs = topic_model.fit_transform(df_pelis["texto"].tolist(), embeddings)

2026-06-14 18:50:01,639 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-14 18:50:10,393 - BERTopic - Dimensionality - Completed ✓
2026-06-14 18:50:10,396 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-14 18:50:12,679 - BERTopic - Cluster - Completed ✓
2026-06-14 18:50:12,686 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-14 18:50:13,029 - BERTopic - Representation - Completed ✓


In [46]:
# Número de tópicos encontrados (excluye -1 = outliers)
n_topics = len(topic_model.get_topics()) - 1
print(f"Tópicos encontrados: {n_topics}")
print(f"Outliers (tópico -1): {topics.count(-1)}")

Tópicos encontrados: 67
Outliers (tópico -1): 2287


In [47]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2287,-1_de_en_un_la,"[de, en, un, la, el, una, que, su, drama, come...",[La niñera. Tras ser despedida de su trabajo y...
1,0,267,0_romance_mujer_matrimonio_se,"[romance, mujer, matrimonio, se, una, su, dram...",[Matrimonio compulsivo. Un hombre recién casad...
2,1,170,1_policía_detective_crimen_asesinato,"[policía, detective, crimen, asesinato, mister...",[Maniac Cop. Un asesino vestido de policía ase...
3,2,148,2_escuela_instituto_teacher_estudiante,"[escuela, instituto, teacher, estudiante, roma...",[Half Nelson. Un profesor de secundaria con ad...
4,3,121,3_alienígena_ficción_ciencia_tierra,"[alienígena, ficción, ciencia, tierra, alieníg...",[V. Unos alienígenas humanoides reptilianos in...
5,4,110,4_aventura_fantasía_el_animación,"[aventura, fantasía, el, animación, rey, famil...","[Beowulf, la leyenda. En una tierra sitiada, B..."
6,5,110,5_title_drama_del_as,"[title, drama, del, as, la, historia, una, ser...",[Felicity. Una joven recién salida del institu...
7,6,99,6_familia_madre_su_padre,"[familia, madre, su, padre, hija, una, drama, ...",[Paso a paso. Dos familias se convierten en un...
8,7,82,7_prisión_crimen_rehén_fuga,"[prisión, crimen, rehén, fuga, banco, un, robo...",[Dos fugitivos. Un atracador estúpido toma a J...
9,8,78,8_música_rock_jazz_banda,"[música, rock, jazz, banda, cantante, pianista...",[Purple Rain. Una historia humana de supervive...


In [48]:
# Documentos más representativos del tópico 1
print("Documentos representativos del tópico 1:\n")
for doc in topic_model.get_representative_docs(1):
    print(doc[:100])
    print("---")

Documentos representativos del tópico 1:

Maniac Cop. Un asesino vestido de policía asesina a inocentes en las calles de Nueva York.. acción, 
---
Manhattan Sur. Un detective de la policía toma medidas contra el crimen organizado en Chinatown desp
---
Policías de Nueva York. Los detectives de la 15ª brigada de la policía de Nueva York investigan homi
---


Para cada usuario: concatenar query + histórico y obtener distribución

In [55]:
def build_user_text(usuario_row, pelis_df):
    """Construye el texto concatenado para un usuario"""
    user_text = usuario_row['query']
    
    for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
        nombre = usuario_row[col]
        peli = pelis_df[pelis_df['name'] == nombre]
        if not peli.empty:
            user_text += " " + peli["texto"].values[0]
    
    return user_text

In [56]:
# Agregar columna de texto a usuarios
usuarios['texto'] = usuarios.apply(lambda row: build_user_text(row, df_pelis), axis=1)

In [57]:
_, users_probs = topic_model.transform(usuarios['texto'].tolist())

print(f"users_probs shape: {users_probs.shape}")  # (14, n_topics)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-14 19:01:53,672 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-06-14 19:01:53,696 - BERTopic - Dimensionality - Completed ✓
2026-06-14 19:01:53,697 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-06-14 19:01:53,701 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-06-14 19:01:53,710 - BERTopic - Probabilities - Completed ✓
2026-06-14 19:01:53,711 - BERTopic - Cluster - Completed ✓


users_probs shape: (14, 67)


## Recomendaciones

In [71]:
# Calcular similitud coseno
scores = cosine_similarity(users_probs, probs)

# Crear máscara de películas ya vistas
scores_filtered = scores.copy()

for i, row in usuarios.iterrows():
    historial_names = [row['pelicula_1'], row['pelicula_2'], row['pelicula_3'], 
                       row['pelicula_4'], row['pelicula_5']]
    
    # Encontrar índices de películas en el historial
    historial_indices = df_pelis[df_pelis['name'].isin(historial_names)].index
    
    # Asignar -inf para que no salgan en top-10
    scores_filtered[i, historial_indices] = -np.inf

# Top-10 por usuario
top10_indices = scores.argsort(axis=1)[:, -10:][:, ::-1]

# Ver resultados
for i, row in usuarios.iterrows():
    print(f"\n{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    for idx in top10_indices[i]:
        pelicula = df_pelis.iloc[idx]
        print(f"  {pelicula['name']} ({int(pelicula['year'])}) — {scores[i, idx]:.4f}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
  El cielo sobre Berlín (1987) — 1.0000
  Tocados por un ángel (1994) — 1.0000
  Una historia diferente (1997) — 1.0000
  Ángeles y demonios (1995) — 1.0000
  El cielo se equivocó (1989) — 1.0000
  Sin noticias de Dios (2001) — 1.0000
  Ángel (2001) — 1.0000
  La mujer del predicador (1997) — 1.0000
  Dogma (2000) — 1.0000
  ¡Tan lejos, tan cerca! (1994) — 1.0000

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
  Una árida estación blanca (1989) — 1.0000
  Tiempo de matar (1996) — 1.0000
  Causa justa (1995) — 1.0000
  El demonio vestido de azul (1995) — 1.0000
  Fantasmas del pasado (1997) — 1.0000
  Justicia poética (1994) — 1.0000
  Protegidos por su enemigo (2008) — 1.0000
  Los 4400 (2005) — 1.0000
  Rosewood (1997) — 1.0000
  Monster's Ball (2002) — 1.0000

Camila (definido)
Query: Una comedia donde la relación

Horrible. Al último le recomienda todo Harry Potter jaj